# Assignment 26 : Groq API Chatbot, RAG  & FastAPI Serving

# PART 1 -- Getting Started with GROQ API

# Task 1: Groq API Setup & Basic Chat
## 1. Create a Groq account and generate an API key.
## 2. Install Groq Python SDK.
## 3. Call a Groq LLM 
## 4. Send a Simple prompt and print the response.

In [4]:

from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()
llm = ChatGroq(
    model="openai/gpt-oss-120b"
)

response = llm.invoke("What is Generative AI?")

print(response.content)


**Generative AI** is a class of artificial‑intelligence systems that can **create new content**—such as text, images, audio, video, code, or even 3‑D models—rather than just analyzing or classifying existing data.

---

## Core Idea

1. **Learn from data** – The model is trained on massive datasets (e.g., books, photos, code repositories).  
2. **Capture patterns** – It learns statistical relationships, structures, and “styles” present in that data.  
3. **Generate new instances** – When prompted, it samples from what it has learned to produce something that **didn’t exist before** but looks (or sounds, or reads) like it belongs to the same domain.

---

## How It Works (at a High Level)

| Step | What Happens |
|------|--------------|
| **Training** | The model sees billions of examples and adjusts millions (or billions) of parameters to minimize prediction error. Common architectures: <br>• **Transformer‑based language models** (e.g., GPT‑4, LLaMA) <br>• **Diffusion models** for imag

# Task 2: Build a Groq Chatbot
## 1. Create a function:
## def groq_chat(prompt: str):
## # returns model response
## 2. Handle system + user messages.
## 3. Test chatbot with multiple queries

In [ ]:

from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()

llm = ChatGroq(
    model="openai/gpt-oss-120b"
)

def groq_chat(prompt: str):
    response = llm.invoke([
        ("system", "You are a helpful assistant."),
        ("human", prompt)
    ])
    return response.content

print(groq_chat("What is Python?"))
print(groq_chat("What is Generative AI?"))
print(groq_chat("Explain Machine Learning"))


**Python** is a high‑level, interpreted programming language known for its readability and simplicity. Created by Guido van Rossum and first released in 1991, Python emphasizes clean, human‑friendly syntax (e.g., using indentation to define code blocks) and a “batteries‑included” philosophy that ships with a large standard library for tasks such as file I/O, networking, web development, data analysis, and more.

### Key characteristics
| Feature | Description |
|---------|-------------|
| **Interpreted** | Code is executed line‑by‑line by the Python interpreter; no separate compilation step is required. |
| **Dynamic typing** | Variable types are determined at runtime, allowing flexible and rapid development. |
| **Readability** | Clear, expressive syntax (e.g., `for i in range(5):` instead of `for (int i = 0; i < 5; i++)`). |
| **Extensive standard library** | Modules for everything from regular expressions (`re`) to HTTP (`http.client`). |
| **Cross‑platform** | Runs on Windows, macO

# PART 2 -- Groq + RAG

# Task 3: Groq - Based RAG Pipeline
## 1. Retrieve relevent document chunks using a vector store.
## 2. Construct a RAG prompt using retrieved contextt.
## 3. Pass the promt to Groq LLM.
## 4. Return grounded answers.

In [5]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

load_dotenv()
# 1. Sample documents
documents = [
    Document(page_content="Python is a programming language used in AI and web development."),
    Document(page_content="RAG stands for Retrieval Augmented Generation. It retrieves relevant information before generating an answer."),
    Document(page_content="Groq provides fast inference for supported large language models.")
]

# 2. Create embeddings and vector store
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(documents, embeddings)

# 3. Connect Groq LLM
llm = ChatGroq(
    model="openai/gpt-oss-120b"
)

# 4. RAG function
def rag_chat(question):
    docs = vectorstore.similarity_search(question, k=2)
    context = "\n".join(doc.page_content for doc in docs)

    prompt = f"""
    Answer using only the context below.
    If the answer is not in the context, say "Information not found."

    Context:
    {context}

    Question: {question}
    """

    response = llm.invoke(prompt)
    return response.content

# 5. Test
print(rag_chat("What is RAG?"))
print(rag_chat("What is Python used for?"))
print(rag_chat("What is the capital of Japan?"))


C:\Users\Anish Rawat\AppData\Local\Temp\ipykernel_18532\1483345417.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
c:\Users\Anish Rawat\GEN_AI\Langchain Projects\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 471.33it/s]


RAG stands for Retrieval Augmented Generation. It retrieves relevant information before generating an answer.
Python is used for AI and web development.
Information not found.


# Task 4:  Prompt Template for Groq RAG
## 1. Create a structured prompt template:
## . System Message
## . Retrieved context
## . User question
## 2. Ensure answers are based only on context.

In [6]:

from langchain_core.prompts import ChatPromptTemplate

load_dotenv()
# 1. Groq model
llm = ChatGroq(
    model="openai/gpt-oss-120b"
)

# 2. Create prompt template
prompt_template = ChatPromptTemplate.from_messages([
    ("system", """
    You are a helpful RAG assistant.
    Answer using ONLY the provided context.
    If the answer is not in the context,
    say: Information not found in the context.
    """),
    ("human", """
    Retrieved Context:
    {context}

    User Question:
    {question}
    """)
])

# 3. Create RAG chain
rag_chain = prompt_template | llm

# 4. Test
context = """
Python is a programming language.
Python is used in AI and web development.
RAG retrieves relevant information from documents
to help generate answers.
"""

question = "What is Python used for?"

response = rag_chain.invoke({
    "context": context,
    "question": question
})

print(response.content)


Python is used in AI and web development.


# PART 3 -- Building API using FastAPI

# Task 5 : Create FastAPI Application
## 1. create a FastAPI app (main.py)
## 2. Add a health check endpoint GET/health
## 3. Add a POST endpoint for chatbot:
## POST / chat
## Request body:
##  {"query":"Your question here"}
## Response:
## { "answer":"model response"}

# Task 6 : FastAPI + Groq Integration
## 1. Inside /Chat, call Groq chatbot or RAG logic.
## 2. Handle errors gracefully.
## 3. Add request validation using Pydantic models

# PART 4 -- Serving the Groq App

# Task 7: Run & Test FastAPI App Locally
## 1. Run using :
## uvicorn main:app --reload
## 2. Test endpoints using:
## . Browser
## . Postman
## . Swagger UI(/docs)

# Task 8 : Production Readiness
## 1. Create requiremnts.txt
## 2. Store API keys using environment variables
## 3. Add basic logging.


# Task 10 : Observations & Insights
# .  Write short answers:
## 1. Why Groq is suitable for real time apps
## 2. Groq vs OpenAI latency comparison
## 3. Benefits of API-first GenAI architecture 

# 1. Why Groq is suitable for real-time applications?
## Groq provides fast LLM inference with low latency. It generates responses quickly, making it suitable for chatbots, AI assistants, and real-time applications.

# 2. Groq vs OpenAI latency comparison
##  Groq: Offers very fast inference and quick response generation.
##  OpenAI: Provides different models with varying response speeds and capabilities.
## Groq can be faster for some workloads, but actual latency depends on the model, prompt size, server load, and network.

# 3. Benefits of API-first GenAI architecture
## . Scalability: Applications can handle increasing user requests.
## .Flexibility: Easily switch between different AI models.
## .Integration: Connect AI models with websites, mobile apps, and other services.
## .Maintainability: Backend and frontend can be developed separately.
## .Reusability: The same API can be used by multiple applications.